In [ ]:
!pip install pandas
!pip install torch
!pip install transformers
!pip install scikit-learn
!pip install numpy
!pip install urllib3
!pip install onnxruntime
!pip install onnx
!pip install thread6

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 53.3 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import torch
import onnx
import onnxruntime
import threading
import multiprocessing
import numpy as np
import pandas as pd
import urllib.request
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup

# Load your dataset by simply uncomment any one of below
url = "<dataset-url>"  # dataset CSV location removed; set before running

#download and read the contents
headers = {'User-Agent': 'Mozilla/5.0'}
request = urllib.request.Request(url, headers=headers)
df = pd.read_csv(urllib.request.urlopen(request))

# Initialize the tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('prajjwal1/bert-tiny', num_labels=2)

# Tokenize the data
tokenized = df['hex'].apply((lambda x: tokenizer.encode(x, add_special_tokens=True, max_length=512, truncation=True)))
tokenized.head()

# Pad the data
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)
padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])
padded.shape

# Create attention masks
attention_mask = np.where(padded != 0, 1, 0)
attention_mask.shape

# Convert to tensors
input_ids = torch.tensor(padded)
attention_mask = torch.tensor(attention_mask)
labels = torch.tensor(df['label'])

# Create the dataset
dataset = TensorDataset(input_ids, attention_mask, labels)

# Split the dataset
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create the dataloaders
batch_size = 16
train_dataloader = DataLoader(train_dataset, sampler=None, batch_size=batch_size)
val_dataloader = DataLoader(val_dataset, sampler=None, batch_size=batch_size)

# Set the optimizer
optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-8)

# Set the number of epochs
epochs = 10
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define the training loop
def train():
    model.train()
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)
        model.zero_grad()
        outputs = model(batch_input_ids, token_type_ids=None, attention_mask=batch_attention_mask, labels=batch_labels)
        loss = outputs[0]
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    avg_train_loss = total_train_loss / len(train_dataloader)
    return avg_train_loss


# Train the model
for epoch in range(epochs):
    train_loss = train()
    model.eval()
    total_val_loss = 0
    predictions, true_labels = [], []
    for batch in val_dataloader:
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)
        with torch.no_grad():
            outputs = model(batch_input_ids, token_type_ids=None, attention_mask=batch_attention_mask)
        logits = outputs[0]
        logits = logits.detach().cpu().numpy()
        label_ids = batch_labels.to('cpu').numpy()
        predictions.append(logits)
        true_labels.append(label_ids)

    flat_predictions = np.concatenate(predictions, axis=0)
    flat_predictions = np.argmax(flat_predictions, axis=1).flatten()
    flat_true_labels = np.concatenate(true_labels, axis=0)
    val_accuracy = accuracy_score(flat_true_labels, flat_predictions)
    print('Epoch:', epoch+1)
    print('Training loss:', train_loss)
    print('Validation accuracy:', val_accuracy)
    print()


with open('/content/drive/My Drive/bert-min.pt', 'wb') as f:
    torch.save(model, f)

# Define the prediction function
def predict(text):
    tokenized = tokenizer.encode(text, add_special_tokens=True)
    padded = np.array([tokenized + [0]*(max_len-len(tokenized))])
    attention_mask = np.where(padded != 0, 1, 0)
    input_ids = torch.tensor(padded)
    attention_mask = torch.tensor(attention_mask)
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    with torch.no_grad():
        outputs = model(input_ids, token_type_ids=None, attention_mask=attention_mask)
    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    predictions = np.argmax(logits, axis=1)
    return predictions[0]

# Define the prediction function
def predict(text):
    tokenized = tokenizer.encode(text, add_special_tokens=True)
    padded = np.array([tokenized + [0]*(max_len-len(tokenized))])
    attention_mask = np.where(padded != 0, 1, 0)
    input_ids = torch.tensor(padded)
    attention_mask = torch.tensor(attention_mask)
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    with torch.no_grad():
        outputs = model(input_ids, token_type_ids=None, attention_mask=attention_mask)
    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    predictions = np.argmax(logits, axis=1)
    if predictions[0] == 0:
        return 'Negative'
    else:
        return 'Positive'

# Test the model with hexadecimals
print(predict('0x04kjfngjfkjngkjnkjnk'))

#with open('/content/drive/My Drive/bert.onnx', 'wb') as f:
#    torch.onnx.export(model, (b_input_ids, b_input_mask), f, opset_version=11, input_names=['input_ids'], output_names=['output'], dynamic_axes={'input_ids': {0: 'batch_size'}, 'output': {0: 'batch_size'}})



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch: 1
Training loss: 0.7144060134887695
Validation accuracy: 1.0

Epoch: 2
Training loss: 0.7151573896408081
Validation accuracy: 1.0

Epoch: 3
Training loss: 0.6945095062255859
Validation accuracy: 1.0

Epoch: 4
Training loss: 0.7410440444946289
Validation accuracy: 1.0

Epoch: 5
Training loss: 0.703952431678772
Validation accuracy: 1.0

Epoch: 6
Training loss: 0.6866757273674011
Validation accuracy: 1.0

Epoch: 7
Training loss: 0.6922234296798706
Validation accuracy: 1.0

Epoch: 8
Training loss: 0.7188328504562378
Validation accuracy: 1.0

Epoch: 9
Training loss: 0.6879843473434448
Validation accuracy: 1.0

Epoch: 10
Training loss: 0.7104601860046387
Validation accuracy: 1.0

Positive
